In [1]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Load UCI promoter dataset directly
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"

df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nLabel counts:")
print(df['label'].value_counts())

Shape: (106, 3)

First few rows:
  label         id                                           sequence
0     +        S10  \t\ttactagcaatacgcttgcgttcggtggttaagtatgtataat...
1     +       AMPC  \t\ttgctatcctgacagttgtcacgctgattggtgtcgttacaat...
2     +       AROH  \t\tgtactagagaactagtgcattagcttatttttttgttatcat...
3     +      DEOP2  \taattgtgatgtgtatcgaagtgtgttgcggagtagatgttagaa...
4     +  LEU1_TRNA  \ttcgataattaactattgacgaaaagctgaaaaccactagaatgc...

Label counts:
label
+    53
-    53
Name: count, dtype: int64


In [2]:
def clean_sequence(seq):
    seq = seq.upper()
    seq = seq.strip()

    return seq
df_cleaned = df.copy()
df_cleaned['label'] = df_cleaned['label'].map({'-': 0, '+': 1})
df_cleaned['sequence'] = df_cleaned['sequence'].apply(clean_sequence)
df_cleaned.head(3)

,label,id,sequence
0,1,S10,TACTAGCAATACGCTTGCGTTCGGTGGTTAAGTATGTATAATGCGC...
1,1,AMPC,TGCTATCCTGACAGTTGTCACGCTGATTGGTGTCGTTACAATCTAA...
2,1,AROH,GTACTAGAGAACTAGTGCATTAGCTTATTTTTTTGTTATCATGCTA...


In [9]:
all_chars = set()
for seq in df_cleaned:
    all_chars.update(seq)
print(f"Unique characters: {all_chars}")

Unique characters: {'G', 'A', 'C', 'T'}


In [6]:
class PromoterDataset(Dataset):
    def __init__(self, sequences, labels, encoder):
        self.sequences = sequences
        self.labels = labels
        self.encoder = encoder
        self.encoded_sequences = [self.encoder.encode(seq).T for seq in self.sequences]
        self.encoded_labels = torch.tensor(self.labels, dtype=torch.float32)
        self.encoded_sequences = torch.stack(self.encoded_sequences) # Transpose to (4,57)
    
    def __len__(self):
        return len(self.encoded_sequences)
    
    def __getitem__(self, idx):
        return self.encoded_sequences[idx], self.encoded_labels[idx]
    
class ExperimentTracker:
    def __init__(self):
        self.runs = {}  # nested dict for storage
    
    def log_run(self, name, hyperparams, fold_results):
        self.runs[name] = {
            'hyperparams': hyperparams,
            'folds': fold_results
        }
    
    def to_dataframe(self):
        """Convert to DataFrame for analysis"""
        rows = []
        for name, data in self.runs.items():
            row = {'experiment': name}
            row.update(data['hyperparams'])  # lr, dropout, etc
            
            accs = [f['accuracy'] for f in data['folds']]
            row['mean_accuracy'] = np.mean(accs)
            row['std_accuracy'] = np.std(accs)
            
            # Individual folds
            for i, fold in enumerate(data['folds']):
                row[f'fold_{i+1}_acc'] = fold['accuracy']
                row[f'fold_{i+1}_train_loss'] = fold['train_loss']
                row[f'fold_{i+1}_test_loss'] = fold['test_loss']
            
            rows.append(row)
        
        return pd.DataFrame(rows).set_index('experiment')

In [ ]:
import wandb

# Start a run
wandb.init(
    project="promoter-cnn-classifier",
    name="batchnorm_experiment",
    config={
        'hidden_size': 96,
        'dropout': 0.5,
        'batchnorm': True,
        'kernel_size': 10,
        'lr': 0.001,
        'epochs': 50
    }
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

In [12]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import sys
sys.path.append('../src/')
from UpdatedSequenceEncoder import SequenceEncoder
from promoter_cnn import PromoterCNNClassifier

# Prepare data
sequences = df_cleaned['sequence'].values
labels = df_cleaned['label'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
fold_losses = []

for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels)):
    print(f"\n{'='*40}")
    print(f"Fold {fold+1}/5")
    print(f"  Train: {len(train_idx)} samples")
    print(f"  Test:  {len(test_idx)} samples")
    
    # 1. Split data using train_idx, test_idx
    train_sequences, train_labels = sequences[train_idx], labels[train_idx]
    test_sequences, test_labels = sequences[test_idx], labels[test_idx]
    
    # 2. Create PromoterDataset for each split
    train_dataset = PromoterDataset(train_sequences, train_labels, SequenceEncoder())
    test_dataset = PromoterDataset(test_sequences, test_labels, SequenceEncoder())

    # 3. Create DataLoaders (batch_size=8)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    # 4. Create FRESH model + optimizer + criterion

    model = PromoterCNNClassifier(4, 96, 1)
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters())
    # 5. Train for 50 epochs
    for epoch in range(50):
        model.train()
        for X_batch, y_batch in train_loader:
            y_batch = y_batch.float()
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
        wandb.log({
        'epoch': epoch,
        'train_loss': loss
    })
    print(f"  Final train loss: {loss:.4f}")

    # 6. Evaluate on test fold
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        final_loss = 0
        for X_batch, y_batch in test_loader:
            predictions = model(X_batch)
            predicted_labels = (predictions > 0.5).float()
            correct += (predicted_labels.squeeze() == y_batch).sum().item()
            total += len(y_batch)
            final_loss += criterion(predictions.squeeze(), y_batch.float()).item()
    print(f"  Final test loss:  {final_loss:.4f}")
    # 7. Append accuracy to fold_accuracies
    fold_losses.append(final_loss / len(test_loader))        
    accuracy = correct / total if total > 0 else 0
    fold_accuracies.append(accuracy)
    wandb.log({
    'mean_accuracy': np.mean(fold_accuracies),
    'std_accuracy': np.std(fold_accuracies),
    'fold': fold + 1
})
wandb.finish()
# If train loss << test loss → overfitting
# If train loss ≈ test loss → good generalization

# Results
print(f"\n{'='*40}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*40}")
print(f"Fold accuracies: {[f'{a:.2%}' for a in fold_accuracies]}")
print(f"Mean accuracy:   {np.mean(fold_accuracies):.2%}")
print(f"Std deviation:   {np.std(fold_accuracies):.2%}")


Fold 1/5
  Train: 84 samples
  Test:  22 samples
  Final train loss: 0.0064
  Final test loss:  0.3079

Fold 2/5
  Train: 85 samples
  Test:  21 samples
  Final train loss: 0.1022
  Final test loss:  0.1524

Fold 3/5
  Train: 85 samples
  Test:  21 samples
  Final train loss: 0.0173
  Final test loss:  0.1557

Fold 4/5
  Train: 85 samples
  Test:  21 samples
  Final train loss: 0.0178
  Final test loss:  0.2108

Fold 5/5
  Train: 85 samples
  Test:  21 samples
  Final train loss: 0.0002
  Final test loss:  0.9952

CROSS-VALIDATION RESULTS
Fold accuracies: ['100.00%', '100.00%', '100.00%', '100.00%', '90.48%']
Mean accuracy:   98.10%
Std deviation:   3.81%


In [13]:
tracker = ExperimentTracker()

# After each K-Fold run, log it:
tracker.log_run(
    name='batchnorm',
    hyperparams={
        'hidden_size': 96,
        'dropout': 0.5,
        'kernel_size': 10,
        'lr': 0.001,
        'epochs': 50,
        'batch_size': 8,
        'batchnorm': True
    },
    fold_results=[
        {'accuracy': 1.0000, 'train_loss': 0.0064, 'test_loss': 0.3079},
        {'accuracy': 1.0000, 'train_loss': 0.1022, 'test_loss': 0.1524},
        {'accuracy': 1.0000, 'train_loss': 0.0173, 'test_loss': 0.1557},
        {'accuracy': 1.0000, 'train_loss': 0.0178, 'test_loss': 0.2108},
        {'accuracy': 0.9048, 'train_loss': 0.0002, 'test_loss': 0.9952},
    ]
)

In [15]:
results_df = tracker.to_dataframe()
print(results_df[['mean_accuracy', 'std_accuracy', 'dropout', 'hidden_size']])
#print(results_df.sort_values('mean_accuracy', ascending=False))

            mean_accuracy  std_accuracy  dropout  hidden_size
experiment                                                   
batchnorm         0.98096       0.03808      0.5           96
